# Moshi A/B Self-Play — 시드 통합 파이프라인

v1과 동일한 검증된 `ab_selfplay.py` + 게이트 순서를 유지하되, 시드를
**Haan 레포의 `SeedPromptDataset`** (HF `mythicinfinity/libriheavy`,
Apache 2.0)으로 생성하고, wav 재인코딩 없이 **Mimi 코드 safetensors를
`ab_selfplay_loop`에 직접 주입**합니다. 결과물은 로컬에서
`python -m project_amnesty.datasets en_kd --stage ingest`로 파이프라인에 투입합니다.

**게이트 순서 (각 게이트를 통과하기 전에 다음 단계로 넘어가지 말 것):**

| 게이트 | 내용 | 통과 기준 |
|---|---|---|
| 0 | stock 단일 인스턴스 데모 | 정상적인 응답 음성 |
| 2 | A/B cross-feed 풀 실행 | 양쪽 발화 + 턴 교대 |
| 3 | 배치 생성 + 통계 → zip 다운로드 → 로컬 ingest | filter 통과 |

*(게이트 1 — seed → A 단독 베이스라인 — 은 검증 완료되어 이 버전에서 제거했습니다.
재현이 필요하면 `generate_dialogue_from_codes(..., handoff_delay_ms=999_999)`로
실행하면 됩니다.)*

v1 대비 바뀐 것: 섹션 2(시드), 게이트 2/배치 셀의 `generate_dialogue_from_codes`
사용, A/B 합본 스테레오 청취, 마지막 ingest 안내. 나머지는 v1 그대로입니다.


## 0. 환경 설치

`NO_CUDA_GRAPH=1`은 **모든 스트리밍 상태 생성 전에** 설정되어야 합니다 (depformer logit 캡처는 CUDA graph 내부에서 불가능 — `moshi/utils/compile.py:170-172`). pip이 런타임 재시작을 요구하면 재시작 후 이 셀부터 다시 실행하세요.

**⚠️ 런타임 유형 변경(L4↔A100 등) = 새 VM = 디스크 초기화입니다.** 클론/설치/seed가 전부 사라지므로 반드시 이 셀부터 순서대로 재실행하세요. 아래 설치 셀은 설치 직후 import 검증을 수행하므로, 설치가 유실된 채 진행하는 실수를 여기서 차단합니다.

**torch 버전 충돌 주의**: moshi는 `torch>=2.2,<2.10`을 핀하는데 (pyproject.toml:14) 최신 Colab 이미지는 torch 2.11+입니다. 일반 설치 시 pip이 torch 다운그레이드(수 GB CUDA 휠)를 시도하다 실패/중단되면 moshi가 설치되지 않은 채 남습니다. 아래 셀은 `--no-deps`로 moshi만 설치하고 소형 의존성을 수동 설치해 **Colab의 torch를 그대로 유지**합니다. 사용 경로 (LMGen 스트리밍/exec_mask/depformer 캡처)는 torch 2.13에서 tiny-모델 검증을 통과했으므로 핀 상한은 이 파이프라인에 한해 무시해도 안전합니다.

In [ ]:
%env NO_CUDA_GRAPH=1
import torch
assert torch.cuda.is_available(), "GPU 런타임이 아닙니다 — 런타임 유형을 변경하세요."
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 2**30
cc = torch.cuda.get_device_capability(0)
print(f"GPU: {name} | {vram:.0f} GB | compute capability {cc}")
assert cc >= (8, 0), (
    f"{name}은 pre-Ampere(bf16 미지원)입니다. T4에서는 실행 불가 — "
    "L4 또는 A100 런타임으로 변경하세요.")

In [ ]:
!git clone -q https://github.com/kyutai-labs/moshi || echo "(clone skipped: directory exists)"
!cd moshi && git log -1 --format="moshi repo @ %h %ad" --date=short
# --no-deps: moshi의 torch<2.10 핀 때문에 pip이 Colab torch(2.11+)를 다운그레이드
# 하려다 실패하는 것을 방지. 필요한 소형 의존성만 직접 설치 (bitsandbytes는 q8용,
# aiohttp/sounddevice는 client/server 전용이라 제외).
!pip install -q --no-deps ./moshi/moshi
!pip install -q einops sentencepiece sphn safetensors tqdm bitsandbytes
!python -c "import moshi.models.lm as m; print('install verified:', m.__file__)"

## 1. 파이프라인 모듈 (검증된 `ab_selfplay.py` 내장)

이 셀이 파일을 그대로 기록하므로 별도 업로드가 필요 없습니다. 헤더 주석에 코드 인스펙션 근거(file:line)가 정리되어 있습니다.

In [ ]:
%%writefile ab_selfplay.py
# =============================================================================
# Moshi A/B Self-Play Soft-Label Generation Pipeline (Colab prototype)
# =============================================================================
# KMoshi project — teacher-side dialogue generation + KD soft-label capture.
#
# Verified against kyutai-labs/moshi @ e6a55d2 (2026-05-16):
#   * LMGen._step always requires the OTHER stream as external input:
#     needed_tokens = num_codebooks - dep_q - 1 = 8   (moshi/models/lm.py:683)
#     Released config n_q=16, dep_q=8 (models/loaders.py:94-95); the depformer
#     has only dep_q=8 output heads (lm.py:231). => The released checkpoint
#     CANNOT free-run the other-speaker stream (paper Sec. 5.6 mode is NOT in
#     the released code/weights). The A/B cross-feed below is therefore the
#     correct (and only) way to get simulated two-party dialogue.
#   * Two LMGen instances CANNOT share one LMModel: the KV cache lives in the
#     LMModel's streaming state (lm.py:652). => A and B are batch items 0 and 1
#     of a SINGLE LMGen with batch_size=2 (weights trivially shared, KV caches
#     independent per batch item).
#   * B's cold start is implemented with the repo's own mechanism:
#     StreamingModule.set_exec_mask (modules/streaming.py:183) +
#     LMGen(support_out_of_sync=True) (lm.py:571,774). While masked, B's
#     offsets/KV are untouched; at handoff B starts from a fresh state.
#   * Cross-feed happens at TOKEN level: A's own-stream output tokens ARE Mimi
#     tokens, so they are fed directly as B's other-stream input (no lossy
#     decode->re-encode). LMGen output is delayed by max_delay=1 frame
#     (delays=[0,0,1*7,0,1*7], loaders.py:118), so the effective acoustic
#     cross-feed lag is 2 frames = 160 ms. Documented, acceptable for a
#     prototype (comparable to network latency in live use).
#   * Depformer (audio) logits have NO hook in stock LMGen — sampling happens
#     inside depformer_step (lm.py:829-838). SoftLabelLMGen below overrides it.
#     REQUIRES NO_CUDA_GRAPH=1 (utils/compile.py:170-172) because
#     depformer_step is wrapped in CUDAGraphed (lm.py:632) and Python-side
#     capture inside a CUDA graph is not possible.
#   * First-frame double-step quirk mirrored from run_inference.py:165-170.
#   * PyTorch checkpoints: bf16 (~24 GB VRAM) or experimental q8
#     (kyutai/moshiko-pytorch-q8, loaders.py:33). int4 exists for MLX ONLY —
#     the plan's "int4" option is infeasible for the PyTorch Colab path.
#
# Colab usage (GPU runtime):
#   %env NO_CUDA_GRAPH=1
#   !git clone https://github.com/kyutai-labs/moshi && pip install -e moshi/moshi
#   from ab_selfplay import *
#   run = load_models()                      # auto: q8 on <24GB, bf16 on A100
#   result = generate_dialogue(run, "seed.wav", handoff_delay_ms=500)
#   save_dialogue(result, "dialogue_000")
#   stats = dialogue_stats(result)
# =============================================================================

import os

# Must be set BEFORE any LMGen streaming state is created (checked per call in
# moshi/utils/compile.py:172, but set it defensively at import time).
os.environ.setdefault("NO_CUDA_GRAPH", "1")

import json
import math
import dataclasses
import typing as tp

import numpy as np
import torch

from moshi.models.lm import LMModel, LMGen


# -----------------------------------------------------------------------------
# 1. Soft-label capturing LMGen
# -----------------------------------------------------------------------------

class SoftLabelLMGen(LMGen):
    """LMGen that captures pre-sampling top-k logits for ALL generated streams.

    Captured per *internal* model step s (the delayed joint-sequence layout of
    Moshi Eq. 6 — exactly the layout a KD student trained on the same delay
    schedule needs; the delay schedule is stored in metadata):
      - text stream logits        [B, text_card] -> top-k
      - depformer audio logits    [B, dep_q, card] -> top-k per codebook
      - sampled tokens for text and all dep_q audio codebooks

    Constraints (asserted): cfg_coef == 1 (CFG doubles the batch and mixes
    logits, lm.py:714-732 — out of scope here), NO_CUDA_GRAPH=1.
    """

    def __init__(self, *args, topk: int = 32, **kwargs):
        kwargs.setdefault("support_out_of_sync", True)
        assert kwargs.get("cfg_coef", 1.0) == 1.0, "CFG not supported by capture."
        assert os.environ.get("NO_CUDA_GRAPH", "") not in ("", "0"), (
            "Set NO_CUDA_GRAPH=1 before creating streaming state: depformer "
            "logit capture cannot run inside a CUDA graph (lm.py:632)."
        )
        # Text-logit capture uses the stock hook (lm.py:734-735).
        assert "on_text_logits_hook" not in kwargs and "on_text_hook" not in kwargs
        kwargs["on_text_logits_hook"] = self._capture_text_logits
        kwargs["on_text_hook"] = self._capture_text_token
        super().__init__(*args, **kwargs)
        self.topk = topk
        self.reset_capture()

    # --- capture buffers ------------------------------------------------------
    def reset_capture(self):
        self.cap_text_vals: list[torch.Tensor] = []   # [B, topk] fp16
        self.cap_text_idx: list[torch.Tensor] = []    # [B, topk] int32
        self.cap_text_tok: list[torch.Tensor] = []    # [B] int32
        self.cap_audio_vals: list[torch.Tensor] = []  # [B, dep_q, topk] fp16
        self.cap_audio_idx: list[torch.Tensor] = []   # [B, dep_q, topk] int32
        self.cap_audio_tok: list[torch.Tensor] = []   # [B, dep_q] int32
        self._dep_vals_step: list[torch.Tensor] = []
        self._dep_idx_step: list[torch.Tensor] = []
        self._dep_tok_step: list[torch.Tensor] = []

    def _capture_text_logits(self, text_logits: torch.Tensor):
        # text_logits: [B, 1, 1, text_card] (lm.py:733)
        lg = text_logits[:, 0, 0].float()
        k = min(self.topk, lg.shape[-1])
        vals, idx = lg.topk(k, dim=-1)
        self.cap_text_vals.append(vals.to(torch.float16).cpu())
        self.cap_text_idx.append(idx.to(torch.int32).cpu())

    def _capture_text_token(self, text_token: torch.Tensor):
        # text_token: [B] (lm.py:745-747)
        self.cap_text_tok.append(text_token.to(torch.int32).cpu())

    # --- depformer override ---------------------------------------------------
    def depformer_step(self, text_token: torch.Tensor,
                       transformer_out: torch.Tensor) -> torch.Tensor:
        """Copy of LMGen.depformer_step (lm.py:809-850) minus CFG branches,
        plus top-k logit capture per codebook."""
        from moshi.utils.sampling import sample_token

        (B,) = text_token.shape
        prev_token = text_token
        lm_model = self.lm_model
        depformer_tokens: list[torch.Tensor] = []
        self._dep_vals_step.clear()
        self._dep_idx_step.clear()
        self._dep_tok_step.clear()
        assert lm_model.depformer is not None
        assert not lm_model.depformer.is_streaming
        with lm_model.depformer.streaming(B):
            for cb_index in range(lm_model.dep_q):
                input_ = prev_token[:, None, None]
                logits = lm_model.forward_depformer(cb_index, input_, transformer_out)
                # capture BEFORE sampling; logits: [B, 1, 1, card]
                lg = logits[:, 0, 0].float()
                k = min(self.topk, lg.shape[-1])
                vals, idx = lg.topk(k, dim=-1)
                self._dep_vals_step.append(vals.to(torch.float16).cpu())
                self._dep_idx_step.append(idx.to(torch.int32).cpu())
                next_token = sample_token(
                    logits.float(), self.use_sampling, self.temp, self.top_k
                )
                assert next_token.shape == (B, 1, 1)
                next_token = next_token[:, 0, 0]
                self._dep_tok_step.append(next_token.to(torch.int32).cpu())
                depformer_tokens.append(next_token)
                prev_token = next_token
        # commit this step's capture: stack codebook dim
        self.cap_audio_vals.append(torch.stack(self._dep_vals_step, dim=1))
        self.cap_audio_idx.append(torch.stack(self._dep_idx_step, dim=1))
        self.cap_audio_tok.append(torch.stack(self._dep_tok_step, dim=1))
        out = torch.stack(depformer_tokens, dim=1)
        assert out.shape == (B, lm_model.dep_q)
        return out

    def stacked_capture(self) -> dict[str, np.ndarray]:
        """Returns arrays with leading time dim T_internal (model steps)."""
        def st(lst):
            return torch.stack(lst, dim=0).numpy() if lst else np.zeros((0,))
        return {
            "text_topk_vals": st(self.cap_text_vals),   # [T, B, k]
            "text_topk_idx": st(self.cap_text_idx),     # [T, B, k]
            "text_sampled": st(self.cap_text_tok),      # [T, B]
            "audio_topk_vals": st(self.cap_audio_vals), # [T, B, dep_q, k]
            "audio_topk_idx": st(self.cap_audio_idx),   # [T, B, dep_q, k]
            "audio_sampled": st(self.cap_audio_tok),    # [T, B, dep_q]
        }


# -----------------------------------------------------------------------------
# 2. A/B self-play loop (model-agnostic core; mimi only touched by the caller)
# -----------------------------------------------------------------------------

@dataclasses.dataclass
class ABResult:
    # own-channel audio tokens actually fed forward, per instance, [T_out, dep_q]
    own_audio_tokens_A: np.ndarray
    own_audio_tokens_B: np.ndarray
    text_tokens_A: np.ndarray
    text_tokens_B: np.ndarray
    capture: dict                      # SoftLabelLMGen.stacked_capture()
    meta: dict
    pcm_A: tp.Optional[np.ndarray] = None  # filled by decode step
    pcm_B: tp.Optional[np.ndarray] = None


def ab_selfplay_loop(
    lm_gen: SoftLabelLMGen,
    seed_codes: torch.Tensor,        # [1, dep_q_user, T_seed] Mimi tokens of seed
    silence_codes: torch.Tensor,     # [1, dep_q_user, 1] Mimi tokens of silence
    handoff_delay_frames: int = 6,   # ~500 ms at 12.5 Hz
    total_frames: int = 750,         # ~60 s at 12.5 Hz
) -> ABResult:
    """Runs the A/B bootstrap self-play. Assumes lm_gen.streaming(2) is ALREADY
    entered by the caller (so tiny-model tests and real runs share this code).

    Batch item 0 = A, batch item 1 = B.
    """
    lm_model = lm_gen.lm_model
    device = lm_model.device
    ungen = lm_model.ungenerated_token_id
    dep_q = lm_model.dep_q
    n_user = lm_model.num_codebooks - dep_q - 1
    assert seed_codes.shape[1] == n_user, (seed_codes.shape, n_user)
    T_seed = seed_codes.shape[2]
    handoff_step = T_seed + handoff_delay_frames

    lm_gen.reset_capture()

    mask_A_only = torch.tensor([True, False], device=device)
    mask_both = torch.tensor([True, True], device=device)
    lm_gen.set_exec_mask(mask_A_only)

    sil = silence_codes.to(device)                      # [1, n_user, 1]
    last_own = {  # latest VALID own-audio tokens per instance, [1, n_user, 1]
        "A": sil.clone(),
        "B": sil.clone(),
    }
    outA_audio, outB_audio, outA_text, outB_text = [], [], [], []
    exec_log: list[list[bool]] = []  # one entry per forward = per capture row
    cur_mask = [True, False]

    inputs_log: list[torch.Tensor] = []  # per forward, [2, n_user] (repro/debug)

    def _step(x):
        exec_log.append(list(cur_mask))
        inputs_log.append(x[:, :, 0].to(torch.int32).cpu())
        return lm_gen.step(x)

    b_active = False
    first_frame = True

    mask_B_only = torch.tensor([False, True], device=device)

    for s in range(total_frames):
        if not b_active and s >= handoff_step:
            # B's analogue of the first-frame double step (run_inference.py:
            # 165-170): B's first executed input would be replaced by init
            # tokens (lm.py:698-702) and never actually heard. Run one extra
            # step with only B executing so its first cross-feed frame lands.
            lm_gen.set_exec_mask(mask_B_only)
            cur_mask = [False, True]
            warm = torch.cat([sil, last_own["A"]], dim=0)
            _step(warm)
            lm_gen.set_exec_mask(mask_both)
            cur_mask = [True, True]
            b_active = True

        # --- assemble other-stream inputs for this step ---
        if s < T_seed:
            in_A = seed_codes[:, :, s : s + 1].to(device)
        else:
            in_A = last_own["B"] if b_active else sil
        in_B = last_own["A"] if b_active else sil
        input_tokens = torch.cat([in_A, in_B], dim=0)   # [2, n_user, 1]

        # --- step (first-frame double-step, run_inference.py:165-170) ---
        if first_frame:
            out = _step(input_tokens)
            first_frame = False
        out = _step(input_tokens)
        if out is None:
            continue
        # out: [2, dep_q+1, 1], rows are ungen while an item is out of sync
        toks = out[:, :, 0]
        for i, name in enumerate(("A", "B")):
            row = toks[i]
            if (row == ungen).any():
                continue  # not yet valid for this item
            audio = row[1 : dep_q + 1]
            last_own[name] = audio[None, :, None]
            (outA_audio if name == "A" else outB_audio).append(
                audio.to(torch.int32).cpu()
            )
            (outA_text if name == "A" else outB_text).append(
                int(row[0].item())
            )

    def np_stack(lst, width):
        if not lst:
            return np.zeros((0, width), dtype=np.int32)
        return torch.stack(lst, dim=0).numpy()

    meta = dict(
        t_seed_frames=T_seed,
        handoff_delay_frames=handoff_delay_frames,
        handoff_step=handoff_step,
        total_frames=total_frames,
        temp=lm_gen.temp, temp_text=lm_gen.temp_text,
        top_k=lm_gen.top_k, top_k_text=lm_gen.top_k_text,
        topk_capture=lm_gen.topk,
        delays=list(lm_model.delays),
        dep_q=dep_q, n_user=n_user,
        crossfeed_lag_frames=2,  # 1 output delay + 1 explicit prev-step feed
        exec_mask_log=exec_log,  # per capture row: which items were executing
    )
    meta["_inputs_log"] = torch.stack(inputs_log, 0).numpy()  # [T,2,n_user]
    return ABResult(
        own_audio_tokens_A=np_stack(outA_audio, dep_q),
        own_audio_tokens_B=np_stack(outB_audio, dep_q),
        text_tokens_A=np.array(outA_text, dtype=np.int32),
        text_tokens_B=np.array(outB_text, dtype=np.int32),
        capture=lm_gen.stacked_capture(),
        meta=meta,
    )


# -----------------------------------------------------------------------------
# 3. Model loading / audio I/O / persistence (GPU-side, used on Colab)
# -----------------------------------------------------------------------------

def load_models(hf_repo: str | None = None, device: str = "cuda", topk: int = 32):
    """Loads Mimi + Moshi. Auto-selects q8 vs bf16 by available VRAM.
    T4 (no bf16, 16 GB) is NOT a supported target for the PyTorch backend —
    use L4/A100 (Colab Pro). Prints the decision before downloading (plan
    Step 0 requirement)."""
    from moshi.models import loaders

    if hf_repo is None:
        free_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
        cc = torch.cuda.get_device_capability(0)
        if free_gb >= 30:
            hf_repo = "kyutai/moshiko-pytorch-bf16"
        else:
            hf_repo = "kyutai/moshiko-pytorch-q8"
        print(f"[target] GPU {torch.cuda.get_device_name(0)} "
              f"({free_gb:.0f} GB, cc {cc}) -> {hf_repo}")
        assert cc >= (8, 0) or "q8" in hf_repo, (
            "bf16 checkpoint on pre-Ampere GPU is unsupported; use q8 or a "
            "better runtime tier.")
    ci = loaders.CheckpointInfo.from_hf_repo(hf_repo)
    mimi = ci.get_mimi(device=device)
    lm = ci.get_moshi(device=device)
    lm_gen = SoftLabelLMGen(lm, topk=topk)
    return dict(checkpoint_info=ci, mimi=mimi, lm=lm, lm_gen=lm_gen,
                device=device)


def encode_audio(run: dict, wav_path: str) -> torch.Tensor:
    """Seed wav -> Mimi tokens [1, 8, T] (mono, resampled to 24 kHz)."""
    import sphn
    mimi = run["mimi"]
    pcm, _ = sphn.read(wav_path, sample_rate=int(mimi.sample_rate))
    pcm = torch.from_numpy(pcm[None, :1]).to(run["device"]).float()
    frame = int(mimi.sample_rate / mimi.frame_rate)
    T = (pcm.shape[-1] // frame) * frame
    with torch.no_grad(), mimi.streaming(1):
        codes = torch.cat(
            [mimi.encode(c) for c in pcm[..., :T].split(frame, dim=-1)], dim=-1)
    return codes


def silence_codes_of(run: dict) -> torch.Tensor:
    mimi = run["mimi"]
    frame = int(mimi.sample_rate / mimi.frame_rate)
    z = torch.zeros(1, 1, frame, device=run["device"])
    with torch.no_grad(), mimi.streaming(1):
        return mimi.encode(z)


def generate_dialogue(run: dict, seed_wav: str, handoff_delay_ms: int = 500,
                      total_seconds: float = 60.0) -> ABResult:
    mimi, lm_gen = run["mimi"], run["lm_gen"]
    fr = mimi.frame_rate  # 12.5
    seed = encode_audio(run, seed_wav)
    sil = silence_codes_of(run)
    with torch.no_grad(), lm_gen.streaming(2):
        res = ab_selfplay_loop(
            lm_gen, seed, sil,
            handoff_delay_frames=max(0, round(handoff_delay_ms / 1000 * fr)),
            total_frames=round(total_seconds * fr),
        )
    res.meta["seed_wav"] = seed_wav
    # decode own channels for listening / stats
    for name in ("A", "B"):
        toks = getattr(res, f"own_audio_tokens_{name}")
        if len(toks) == 0:
            continue
        t = torch.from_numpy(toks.T[None]).long().to(run["device"])
        with torch.no_grad(), mimi.streaming(1):
            pcm = mimi.decode(t)
        setattr(res, f"pcm_{name}", pcm[0, 0].cpu().numpy())
    return res


def save_dialogue(res: ABResult, path_prefix: str):
    np.savez_compressed(
        path_prefix + ".npz",
        own_audio_tokens_A=res.own_audio_tokens_A,
        own_audio_tokens_B=res.own_audio_tokens_B,
        text_tokens_A=res.text_tokens_A,
        text_tokens_B=res.text_tokens_B,
        **{f"cap_{k}": v for k, v in res.capture.items()},
    )
    with open(path_prefix + ".meta.json", "w") as f:
        json.dump(res.meta, f, indent=2)
    if res.pcm_A is not None:
        import sphn
        sphn.write_wav(path_prefix + "_A.wav", res.pcm_A, 24000)
        sphn.write_wav(path_prefix + "_B.wav", res.pcm_B, 24000)


# -----------------------------------------------------------------------------
# 4. Step 5 sanity statistics
# -----------------------------------------------------------------------------

def dialogue_stats(res: ABResult, frame_ms: float = 80.0,
                   silence_db: float = -40.0, sr: int = 24000) -> dict:
    """Silence ratio, switch events, utterance lengths, overlap — per channel,
    computed on decoded PCM with a simple RMS gate."""
    def voiced_frames(pcm):
        n = int(sr * frame_ms / 1000)
        T = (len(pcm) // n) * n
        fr = pcm[:T].reshape(-1, n)
        rms = np.sqrt((fr ** 2).mean(axis=1) + 1e-12)
        return 20 * np.log10(rms + 1e-12) > silence_db

    out: dict = {}
    va = vb = None
    for name in ("A", "B"):
        pcm = getattr(res, f"pcm_{name}")
        if pcm is None:
            out[name] = None
            continue
        v = voiced_frames(pcm)
        if name == "A":
            va = v
        else:
            vb = v
        runs = np.diff(np.flatnonzero(np.diff(np.r_[0, v.view(np.int8), 0])))
        utt = runs[::2] * frame_ms / 1000 if len(runs) else np.array([])
        out[name] = dict(
            silence_ratio=float(1 - v.mean()),
            n_utterances=int(len(utt)),
            utt_len_mean_s=float(utt.mean()) if len(utt) else 0.0,
            utt_len_max_s=float(utt.max()) if len(utt) else 0.0,
        )
    if va is not None and vb is not None:
        L = min(len(va), len(vb))
        va, vb = va[:L], vb[:L]
        active = np.where(va & ~vb, 0, np.where(vb & ~va, 1, -1))
        speech = active >= 0
        switches = int(np.sum(np.diff(active[speech]) != 0)) if speech.any() else 0
        out["joint"] = dict(
            overlap_ratio=float((va & vb).mean()),
            both_silent_ratio=float((~va & ~vb).mean()),
            n_channel_switches=switches,
        )
    # degenerate-output flags (plan Step 5)
    for name in ("A", "B"):
        toks = getattr(res, f"own_audio_tokens_{name}")
        if len(toks) > 8:
            rep = float((toks[1:] == toks[:-1]).all(axis=1).mean())
            out.setdefault("flags", {})[f"{name}_repeated_frame_ratio"] = rep
    return out


## 2. Seed 준비 — Haan `SeedPromptDataset` (HF Libriheavy)

v1의 LibriSpeech `seed_prep.py` 대신, 학습 레포의 시드 생성기를 그대로 사용합니다.
이렇게 하면 **여기서 만든 시드 = 나중에 통합 엔진이 쓸 시드**가 되어 형식 드리프트가
없습니다.

| 항목 | 내용 |
|---|---|
| 소스 | `mythicinfinity/libriheavy` (스트리밍, 게이트 없음, **Apache 2.0**) |
| 필터/가공 | 6초 미만 스킵 → 앞 10초 → 모노/24 kHz → **Mimi 인코딩** |
| 산출물 | `Haan/data/seed_prompts/{uid}.safetensors` — codes `(8, T)` int16 |
| torchcodec 우회 | `Audio(decode=False)` + soundfile 직접 디코딩 (v1과 같은 이유) |

**전제**: 레포의 최신 커밋(SeedPromptDataset 포함)이 GitHub에 push되어 있어야 합니다.
레포 pip 설치는 하지 않습니다 — pyproject의 transformers fork 핀이 Colab torch와
충돌할 수 있어, 클론된 디렉토리에서 CLI로 직접 실행합니다.

**데이터셋이 실제 적용됐는지 확인하는 법:** 아래 청취 셀에서 시드가 *자연스러운
낭독체 영어*로 들리면 시드 데이터셋은 정상 적용된 것입니다. 시드는 정상인데
생성 음성만 기계음이면 데이터셋 문제가 아니라 생성 쪽(q8 양자화, 반복 붕괴,
B cold-start) 문제입니다 — 게이트 0의 `out_stock.wav`와 음질을 비교하세요.


In [ ]:
!rm -rf Haan

In [ ]:
!pip install -q datasets soundfile safetensors
!git clone -q https://github.com/latentforge/Haan || echo "(clone skipped: directory exists)"
import os
assert os.path.exists("Haan/project_amnesty/datasets/offline/sources/seed_prompt_dataset.py"), (
    "레포에 SeedPromptDataset이 없습니다 — 로컬 커밋을 git push 했는지 확인하세요.")

# --- HF_TOKEN 등록 ------------------------------------------------------------
# 1) https://huggingface.co/settings/tokens 에서 "Read" 권한 토큰 생성
# 2) Colab 왼쪽 사이드바 열쇠(🔑) 아이콘 → "새 보안 비밀 추가"
#    → 이름: HF_TOKEN, 값: 토큰 붙여넣기 → "노트북 액세스" 토글 ON
# 3) 이 셀을 다시 실행 → 아래 whoami 출력으로 로그인 확인
# libriheavy는 공개 데이터셋이라 토큰 없이도 받아지지만, 익명 rate-limit에
# 걸리면 스트리밍이 끊겨 시드가 모자랄 수 있습니다.

# 하드코딩 방식: 아래 따옴표 안에 "hf_..." 토큰을 붙여넣으세요. 비워두면 건너뜁니다.
# (⚠ 이 노트북을 커밋/공유하기 전에 반드시 지울 것 — GitHub 등에 공개되면
#  HF가 토큰을 자동 폐기합니다. Read 권한 토큰 권장.)
HF_TOKEN_HARDCODED = ""
if HF_TOKEN_HARDCODED:
    os.environ["HF_TOKEN"] = HF_TOKEN_HARDCODED

if not os.environ.get("HF_TOKEN"):
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        print("⚠ HF_TOKEN 미설정 — 익명으로 진행합니다 (rate-limit 가능)")
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import whoami
    print("HF 로그인 확인:", whoami()["name"])

# 익명 접속이면 HF가 'Server disconnected'로 연결을 끊는 일이 잦아 스트리밍이
# 중간에 죽을 수 있음 → 최대 3회 재시도 (이미 인코딩된 시드는 캐시로 건너뛰므로
# 재시도가 이어받아 목표 개수까지 채웁니다).
import glob, subprocess, sys

def n_seeds():
    return len(glob.glob("Haan/data/seed_prompts/*.safetensors"))

for attempt in range(3):
    if n_seeds() >= 6:
        break
    print(f"--- seed 생성 시도 {attempt + 1}/3 (현재 {n_seeds()}개) ---")
    # capture 후 print: Windows 로컬 Jupyter에서는 자식 프로세스 출력이
    # 셀에 안 보이므로, 어디서 실행하든 로그가 보이게 함
    r = subprocess.run(
        [sys.executable, "-m", "project_amnesty.datasets", "seed_prompts",
         "--seed-source", "libriheavy", "--out-dir", "data",
         "--limit", "6", "--device", "cuda", "--batch-size", "4"],
        cwd="Haan", capture_output=True, text=True)
    print(r.stdout)
    if r.stderr:
        print(r.stderr)
    if r.returncode != 0:
        print(f"⚠ 종료 코드 {r.returncode} — 위 트레이스백 확인. "
              "일시적 네트워크 오류면 다음 재시도가 이어받습니다.")

SEEDS = sorted(glob.glob("Haan/data/seed_prompts/*.safetensors"))
print(f"{len(SEEDS)} seeds:", *[os.path.basename(p) for p in SEEDS], sep="\n  ")
assert len(SEEDS) >= 3, (
    "3회 시도 후에도 시드가 부족합니다. 위 로그의 트레이스백이 진짜 원인입니다 — "
    "'Server disconnected'/'429'류면 HF_TOKEN 미적용(위 whoami 출력 확인), "
    "그 외(ImportError 등)면 트레이스백을 공유해 주세요.")


In [ ]:
# 시드 청취/게이트0용 wav 디코딩 (Mimi만 로드 — 소형, 7B 본체 아님)
import os, torch, numpy as np, sphn
from safetensors.numpy import load_file
from moshi.models import loaders

_ci = loaders.CheckpointInfo.from_hf_repo("kyutai/moshiko-pytorch-bf16")
_mimi = _ci.get_mimi(device="cuda")
os.makedirs("seeds_wav", exist_ok=True)
SEED_WAVS = []
for p in SEEDS:
    codes = torch.from_numpy(load_file(p)["codes"].astype(np.int64))[None].cuda()
    with torch.no_grad(), _mimi.streaming(1):
        pcm = _mimi.decode(codes)
    out = f"seeds_wav/{os.path.basename(p).removesuffix('.safetensors')}.wav"
    sphn.write_wav(out, pcm[0, 0].cpu().numpy(), 24000)
    SEED_WAVS.append(out)
del _mimi, _ci
torch.cuda.empty_cache()
print("decoded:", *SEED_WAVS, sep="\n  ")


In [ ]:
# 청취 확인 (seed 자체가 정상인지 — 낭독체 영어 발화여야 함)
from IPython.display import Audio, display
for p in SEED_WAVS:
    print(p); display(Audio(p))


In [ ]:
# (선택) Audio-FLAN 시드 추가 — 게이트 데이터셋: 먼저 웹에서 약관 동의 필요
# https://huggingface.co/datasets/HKUSTAudio/Audio-FLAN-Dataset ("Agree and access")
import os
from huggingface_hub import HfApi, hf_hub_download, login

if not os.environ.get("HF_TOKEN"):
    login()   # 토큰 프롬프트 (또는 Colab 보안비밀에 HF_TOKEN 등록)

REPO = "HKUSTAudio/Audio-FLAN-Dataset"
files = HfApi().list_repo_files(REPO, repo_type="dataset")

# 1) speech 도메인 오디오 파일 탐색 — 구조를 먼저 눈으로 확인
speech = sorted(f for f in files if f.startswith("audio_files/") and "speech" in f.lower())
print(f"speech audio files: {len(speech)}")
for f in speech[:15]:
    print(" ", f)
assert speech, "speech 오디오가 안 보입니다 — 게이트 승인 여부와 위 목록을 확인하세요."

# 2) 소량만 다운로드 (시드는 몇 개면 충분 — tar면 풀고, 개별 파일이면 복사)
import shutil, tarfile
os.makedirs("audio_flan_speech", exist_ok=True)
for f in speech[:2]:
    p = hf_hub_download(REPO, f, repo_type="dataset")
    if p.endswith((".tar", ".tar.gz", ".tgz")):
        with tarfile.open(p) as t:
            t.extractall("audio_flan_speech")
    else:
        shutil.copy(p, "audio_flan_speech/")

# 3) 같은 시드 폴더에 누적 (uid가 소스별로 달라 Libriheavy 시드와 공존)
!cd Haan && python -m project_amnesty.datasets seed_prompts --seed-source local --root ../audio_flan_speech --out-dir data --limit 6 --device cuda

import glob
SEEDS = sorted(glob.glob("Haan/data/seed_prompts/*.safetensors"))
print(f"seeds total (libriheavy + audio_flan): {len(SEEDS)}")


## 3. 게이트 0 — stock 단일 인스턴스 데모

**이게 정상 동작하기 전에는 아래로 내려가지 마세요** (계획서 Step 0).
실패하면 환경/체크포인트 문제이며 파이프라인 코드와 무관합니다.

L4(24GB)는 q8, A100은 bf16을 사용합니다. 체크포인트 다운로드는 최초 1회
수 분 소요됩니다.

In [ ]:
import torch
vram = torch.cuda.get_device_properties(0).total_memory / 2**30
STOCK_REPO = "kyutai/moshiko-pytorch-bf16" if vram >= 30 else "kyutai/moshiko-pytorch-q8"
print("using", STOCK_REPO)
# --batch-size 1 필수: 기본값 8이면 out_stock-0..7.wav로 저장되고
# (run_inference.py:306-310) VRAM도 8배 사용 — L4에서는 OOM 위험.
!python -m moshi.run_inference {SEED_WAVS[0]} out_stock.wav --batch-size 1 --hf-repo {STOCK_REPO}

In [ ]:
import glob, os
from IPython.display import Audio, display
# batch-size>1로 돌렸다면 out_stock-<i>.wav로 저장됨 — 둘 다 처리
candidates = (["out_stock.wav"] if os.path.exists("out_stock.wav")
              else sorted(glob.glob("out_stock-*.wav")))
assert candidates, (
    "출력 wav가 없습니다 — 위 셀의 run_inference 로그에서 오류(OOM 등)를 확인하세요. "
    "생성 자체가 실패한 것이므로 게이트 0 미통과입니다.")
print("게이트 0 확인: seed에 대한 정상적인 영어 응답 음성이 들려야 합니다.")
print("파일:", candidates[0])
display(Audio(candidates[0]))

## 4. 모델 로드

`load_models()`는 VRAM 기준으로 q8/bf16을 자동 선택하고 **다운로드 전에 결정을
출력**합니다. 게이트 0에서 이미 받은 체크포인트를 재사용하므로 추가 다운로드는
없습니다.

In [ ]:
# --- moshi import 상태 정화 + 진단 -------------------------------------
# 이전에 설치 없이 import를 시도했다면 sys.modules에 네임스페이스 패키지가
# 캐시되어 있음 → 정화 후 정식 설치본으로 재-import
import sys
for m in [k for k in list(sys.modules) if k == "moshi" or k.startswith("moshi.")]:
    del sys.modules[m]
import moshi
assert hasattr(moshi, "__file__") and moshi.__file__, (
    "moshi가 네임스페이스 패키지로 잡혔습니다 = pip 설치가 이 VM에 없습니다. "
    "섹션 0의 설치 셀을 재실행하세요 (런타임 유형을 바꿨다면 새 VM이라 "
    "클론/설치/seed 전부 재실행 필요).")
print("moshi @", moshi.__file__)

import importlib, ab_selfplay
importlib.reload(ab_selfplay)          # 모듈 수정 후 재실행 대비
from ab_selfplay import (load_models, generate_dialogue, save_dialogue,
                         dialogue_stats)
import torch
run = load_models()
torch.cuda.reset_peak_memory_stats()
print(f"load peak: {torch.cuda.max_memory_allocated()/2**30:.1f} GB")

In [ ]:
# --- 브리지: Mimi 코드 safetensors -> ab_selfplay_loop 직접 주입 ---------------
# generate_dialogue()와 동일하되 wav 인코딩 단계만 생략 (ab_selfplay_loop는 원래
# seed_codes [1, 8, T] 텐서를 받는다 — encode_audio는 호출부 헬퍼일 뿐).
import numpy as np
import torch
from safetensors.numpy import load_file
from ab_selfplay import ab_selfplay_loop, silence_codes_of


def generate_dialogue_from_codes(run, st_path: str, handoff_delay_ms: int = 500,
                                 total_seconds: float = 60.0):
    mimi, lm_gen = run["mimi"], run["lm_gen"]
    fr = mimi.frame_rate  # 12.5
    seed = torch.from_numpy(load_file(str(st_path))["codes"].astype(np.int64))[None]
    sil = silence_codes_of(run)
    with torch.no_grad(), lm_gen.streaming(2):
        res = ab_selfplay_loop(
            lm_gen, seed.to(run["device"]), sil,
            handoff_delay_frames=max(0, round(handoff_delay_ms / 1000 * fr)),
            total_frames=round(total_seconds * fr),
        )
    res.meta["seed_wav"] = str(st_path)   # ingest가 seed_prompt_id로 사용
    for name in ("A", "B"):
        toks = getattr(res, f"own_audio_tokens_{name}")
        if len(toks) == 0:
            continue
        t = torch.from_numpy(toks.T[None]).long().to(run["device"])
        with torch.no_grad(), mimi.streaming(1):
            pcm = mimi.decode(t)
        setattr(res, f"pcm_{name}", pcm[0, 0].cpu().numpy())
    return res


def dialogue_stereo(res):
    """A/B를 하나의 스테레오 트랙으로 합본 (왼쪽=A, 오른쪽=B).

    A는 루프 시작부터, B는 handoff 이후부터 매 스텝 프레임을 내고 둘 다
    마지막 스텝에서 끝나므로, 끝 기준으로 정렬(B 앞쪽을 무음 패딩)하면
    실제 대화 타임라인과 일치합니다."""
    a = res.pcm_A if res.pcm_A is not None else np.zeros(1, dtype=np.float32)
    b = res.pcm_B if res.pcm_B is not None else np.zeros(1, dtype=np.float32)
    L = max(len(a), len(b))
    pad = lambda x: np.pad(x, (L - len(x), 0))
    return np.stack([pad(a), pad(b)])   # [2, L]


print("generate_dialogue_from_codes / dialogue_stereo ready")


## 5. 게이트 2 — A/B cross-feed 풀 실행

- A = 배치 아이템 0, B = 배치 아이템 1 (단일 `LMGen(batch_size=2)`, 가중치 자동 공유)
- 핸드오프: seed 종료 + 500 ms (`handoff_delay_ms=500`) — 첫 버전은 이 값 고정,
  {0, 500, 1000} 스윕은 이후 단계 (계획서 non-goal)
- cross-feed 실효 지연: 2프레임 = 160 ms (출력 딜레이 1 + 명시적 prev-step 1) —
  메타데이터에 기록됨
- batch=2라 KV 캐시가 2배이므로 첫 실행은 30초로 메모리 확인 후 60초로 확장

In [ ]:
torch.cuda.reset_peak_memory_stats()
res = generate_dialogue_from_codes(run, SEEDS[0], handoff_delay_ms=500, total_seconds=30)
print(f"peak: {torch.cuda.max_memory_allocated()/2**30:.1f} GB "
      f"(VRAM 한도 대비 여유 확인 — 여유 있으면 다음 셀부터 60초로)")
print("A frames:", res.own_audio_tokens_A.shape,
      "| B frames:", res.own_audio_tokens_B.shape)

In [ ]:
import json
stereo = dialogue_stereo(res)   # [2, L] — 왼쪽=A, 오른쪽=B, 타임라인 정렬됨
print("=== A/B 대화 합본 (스테레오: 왼쪽=A, 오른쪽=B) ===")
display(Audio(stereo, rate=24000))
print("=== A 채널 단독 ==="); display(Audio(res.pcm_A, rate=24000))
print("=== B 채널 단독 ==="); display(Audio(res.pcm_B, rate=24000))
print(json.dumps(dialogue_stats(res), indent=2))
print("게이트 2 확인: 양쪽 모두 발화, 어느 쪽도 silence_ratio가 0.95+가 아니어야 합니다.")


In [ ]:
# (선택) 텍스트 스트림 확인 — inner monologue가 대화답게 흐르는지
tok = run["checkpoint_info"].get_text_tokenizer()  # loaders.py:315
for name, toks in (("A", res.text_tokens_A), ("B", res.text_tokens_B)):
    txt = "".join(tok.id_to_piece(int(t)) for t in toks if int(t) not in (0, 3))
    print(f"[{name}]", txt.replace("\u2581", " ")[:500])

## 6. Step 5 — 배치 생성 + 통계 (3–5개 seed)

soft label(`cap_audio_topk_vals` 등)과 `exec_mask_log`가 `.npz`에 저장됩니다 —
KD 타깃 추출 시 어떤 캡처 스텝이 어느 인스턴스에 유효한지 필터링에 사용.

용량 감각: top-k=32 기준 약 3 MB/분/대화.

In [ ]:
import numpy as np, json
def save_dialogue(res, path_prefix):
    meta_json, npz_extra = {}, {}
    for k, v in res.meta.items():
        (npz_extra if isinstance(v, np.ndarray) else meta_json)[
            "meta_" + k.lstrip("_") if isinstance(v, np.ndarray) else k] = v
    np.savez_compressed(path_prefix + ".npz",
        own_audio_tokens_A=res.own_audio_tokens_A,
        own_audio_tokens_B=res.own_audio_tokens_B,
        text_tokens_A=res.text_tokens_A, text_tokens_B=res.text_tokens_B,
        **{f"cap_{k}": v for k, v in res.capture.items()}, **npz_extra)
    with open(path_prefix + ".meta.json", "w") as f:
        json.dump(meta_json, f, indent=2)
    if res.pcm_A is not None:
        import sphn, soundfile
        sphn.write_wav(path_prefix + "_A.wav", res.pcm_A, 24000)
        sphn.write_wav(path_prefix + "_B.wav", res.pcm_B, 24000)
        # 합본 스테레오 (왼쪽=A, 오른쪽=B) — soundfile은 (frames, channels) 순서
        soundfile.write(path_prefix + "_stereo.wav", dialogue_stereo(res).T, 24000)

In [ ]:
import json, os
os.makedirs("dialogues", exist_ok=True)
all_stats = {}
for i, st in enumerate(SEEDS):
    r = generate_dialogue_from_codes(run, st, handoff_delay_ms=500, total_seconds=60)
    prefix = f"dialogues/dialogue_{i:03d}"
    save_dialogue(r, prefix)
    key = os.path.basename(st)
    all_stats[key] = dialogue_stats(r)
    print(f"--- {key} -> {prefix}[.npz/.meta.json/_A.wav/_B.wav/_stereo.wav]")
    print(json.dumps(all_stats[key], indent=2))
with open("dialogues/all_stats.json", "w") as f:
    json.dump(all_stats, f, indent=2)


In [ ]:
# 청취 확인 (계획서 Step 5: 반드시 직접 들을 것) — 합본 스테레오 (왼쪽=A, 오른쪽=B)
for i in range(len(SEEDS)):
    p = f"dialogues/dialogue_{i:03d}_stereo.wav"
    print(p); display(Audio(p))
# 채널별 단독 확인이 필요하면 dialogue_XXX_A.wav / _B.wav도 함께 저장되어 있습니다.


In [ ]:
# 판정 기준 요약 출력
for wav, s in all_stats.items():
    j, flags = s.get("joint", {}), s.get("flags", {})
    verdict = []
    for ch in ("A", "B"):
        sr = s[ch]["silence_ratio"] if s[ch] else 1.0
        if sr > 0.9: verdict.append(f"{ch} 거의 무발화(silence {sr:.2f})")
        if flags.get(f"{ch}_repeated_frame_ratio", 0) > 0.5:
            verdict.append(f"{ch} 반복 붕괴 의심")
    if j.get("n_channel_switches", 0) < 3: verdict.append("턴 교대 거의 없음")
    if j.get("overlap_ratio", 0) == 0.0: verdict.append("오버랩 0 — half-duplex 퇴화 신호")
    print(f"{wav}: {'; '.join(verdict) if verdict else 'OK — 대략적 대화 통계 충족'}")

In [ ]:
# 결과 다운로드
!zip -qr dialogues.zip dialogues
from google.colab import files as _f
_f.download("dialogues.zip")

## 7. 로컬 ingest — 파이프라인 투입

다운로드한 `dialogues.zip`을 풀고 Haan 레포에서:

```bash
python -m project_amnesty.datasets en_kd --stage ingest --root <dialogues 폴더> \
    --out-dir data/generated --filter-config configs/data/filter.yaml
```

- 어댑터가 A/B 길이 정렬(시드 인트로 폐기) + top-k 캡처의 delay 해제(샘플 토큰
  완전 일치로 자기검증)를 수행하고, filter.yaml 설정으로 품질 필터까지 실행합니다.
- **filter.yaml의 `text_pad_id`는 student(Qwen3) 기준 `151669`**. ingest가 저장 전에
  텍스트를 student 어휘로 재토큰화(SeqKD)하므로 filter가 보는 PAD는 student 값이어야 한다.
  (teacher Helium PAD는 `3`이지만 그 값을 filter.yaml에 넣으면 전 코퍼스가 never_silent로 거부된다.)
- 이후: `python -m project_amnesty.datasets.offline.prepare_dataset --config configs/data/prepare.yaml`
  → Arrow → 학습.


## 8. 실패 시 분기

| 증상 | 해석 | 조치 |
|---|---|---|
| 게이트 0 실패 | 환경/체크포인트 문제 | 파이프라인과 무관 — 런타임/설치 재확인 |
| 게이트 2에서 A 붕괴 | silence 토큰 공급 방식 문제 가능 | `handoff_delay_ms=999_999`로 A 단독 베이스라인(구 게이트 1)을 재현해 `meta`와 오디오 첨부해 공유 |
| 게이트 2에서 **B만** 퇴화 (첫 응답 불연속 등) | 계획서에 유보해둔 B cold-start KV silence-padding 문제 발현 | 그때 padding 수정 진행 (지금은 구현하지 않음 — 계획서 non-goal) |
| 전반적 기계음 | 시드 문제 아님(섹션 2에서 시드 청취로 분리 확인) — q8 양자화/붕괴/샘플링 쪽 | 게이트 0 `out_stock.wav`와 비교; L4(q8)라면 A100(bf16)으로 재실행해 비교 |
| OOM (L4 + 60초) | batch=2 KV 캐시 | `total_seconds=30`으로, 또는 A100 런타임 |
| 오버랩 정확히 0 | half-duplex 퇴화 | `handoff_delay_ms`/temperature 조정 전에 먼저 여러 seed로 재현성 확인 |

**다음 단계 (이 노트북 범위 밖, 계획서 non-goal):** handoff {0,500,1000} ms 스윕,
KV silence-padding ablation, Qwen3-8B student 학습 루프.
